# Rendering the chosen models at scale

The earlier settings sweep picked its configurations from ten images each, which is enough for a
CLIP score but far too few for FID or Inception Score. This notebook renders the same 500 COCO
captions through five configurations: the Stable Diffusion 1.5 baseline, the tuned sampler, the
tuned sampler with clip skip 2, LDM-BERT and aMUSEd. Nothing is tuned or trained here; the images
are scored in a separate evaluation notebook.

## 1. Setup

Captions are read straight from the attached COCO Dataset, and images, plots and tables go to
`/kaggle/working/outputs/`. The model weights download from the Hugging Face Hub, so the session
needs internet access.

In [ ]:
import gc, os, textwrap, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import diffusers
import transformers
from diffusers import (AmusedPipeline, EulerAncestralDiscreteScheduler, LDMTextToImagePipeline,
                       PNDMScheduler, StableDiffusionPipeline)
from transformers import CLIPTextModel, CLIPTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
diffusers.logging.set_verbosity_error()


# The Dataset is read in place from /kaggle/input; nothing is copied or written under it.
def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        return found[0].parent, Path("/kaggle/working/outputs")
    for root in (Path("."), Path("..")):
        if (root / "data" / "processed" / "pairs.parquet").is_file():
            return root / "data" / "processed", root / "outputs" / "milestone_3"
    raise RuntimeError("data/processed/pairs.parquet not found -- run from the repo root")


PROCESSED, OUT = resolve_paths()
IMAGES, PLOTS, TABLES = OUT / "images", OUT / "plots", OUT / "tables"
for d in (IMAGES, PLOTS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"

print(f"torch {torch.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
print(f"device  : {DEVICE}  {GPU_NAME}")
if DEVICE.type == "cuda":
    print(f"memory  : {torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.1f} GB")
print(f"data    : {PROCESSED}")
print(f"outputs : {OUT}")

### Settings

`M3_SMOKE=1` in the environment renders 8 prompts instead of 500, for a quick end-to-end check.
The three SD configurations differ only in sampler, step count and which CLIP layer conditions the
UNet. Batch size 4 is what the earlier runs used on both an 8 GB laptop GPU and a Kaggle T4.

In [ ]:
SEED = 42
SMOKE = os.environ.get("M3_SMOKE", "0") == "1"
N_PROMPTS = 8 if SMOKE else 500
BATCH_SIZE = 4
GUIDANCE = 7.5
CLIP_MAX_TOKENS = 77

SD_REPOS = ["stable-diffusion-v1-5/stable-diffusion-v1-5",
            "sd-legacy/stable-diffusion-v1-5",
            "CompVis/stable-diffusion-v1-4"]
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"
SD_SIZE = 512
TIMESTEP_SPACING = "leading"
SD_CONFIGS = {
    "sd_baseline":        {"sampler": "pndm",    "steps": 50, "layer": "last"},
    "sd_tuned":           {"sampler": "euler_a", "steps": 30, "layer": "last"},
    "sd_tuned_clipskip2": {"sampler": "euler_a", "steps": 30, "layer": "penultimate"},
}

LDM_REPO, LDM_STEPS, LDM_SIZE = "CompVis/ldm-text2im-large-256", 50, 256
AMUSED_REPO, AMUSED_STEPS = "amused/amused-256", 12

CONFIG_ORDER = [*SD_CONFIGS, "ldm_bert", "amused"]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"smoke {SMOKE} | {N_PROMPTS} prompts | batch {BATCH_SIZE} | CFG {GUIDANCE} | {DTYPE}")

## 2. The prompt set

One caption per image (the first-ranked one), 500 images sampled with a fixed seed. The guidance
notebook samples the same way and takes the first 200, so the two share prompts. The table written
here maps each image index back to its COCO image for the evaluation notebook.

In [ ]:
pairs = pd.read_parquet(PROCESSED / "pairs.parquet")
sample = pairs[pairs.cap_rank == 0].sample(n=500, random_state=42).reset_index(drop=True)
sample["idx"] = range(len(sample))

prompts = sample.head(N_PROMPTS).copy()
prompts["image_512_path"] = prompts["image_512_path"].map(lambda p: Path(p).name)
prompts[["idx", "image_id", "caption", "image_512_path"]].to_csv(TABLES / "m3_prompts.csv", index=False)

IDX = prompts["idx"].tolist()
CAPTIONS = prompts["caption"].tolist()
assert IDX == list(range(len(IDX)))

print(f"{len(pairs):,} pairs -> {len(sample)} sampled -> {len(prompts)} rendered here")
print(f"wrote {TABLES / 'm3_prompts.csv'}\n")
for i, caption in zip(IDX[:5], CAPTIONS[:5]):
    print(f"  {i:03d}  {caption}")

## 3. Helpers

Every prompt gets its own CPU generator seeded `SEED + idx`, used for its starting noise and for any
noise the sampler draws along the way. A prompt's image therefore does not depend on the batch it
lands in, and every SD configuration starts prompt *i* from the same latent.

Rendering is resumable: an image already on disk is skipped, PNGs are written under a temporary
name and renamed once complete, and the log is rewritten after every configuration. After a
resumed run the timing columns cover only the images rendered in the latest session.

In [ ]:
LOG_PATH = TABLES / "m3_generation_log.csv"
LOG_COLUMNS = ["config", "model", "encoder", "sampler", "steps", "guidance", "resolution",
               "n_images", "seconds_total", "sec_per_image", "peak_vram_gb"]
log_rows = ({row["config"]: row for row in pd.read_csv(LOG_PATH).to_dict("records")}
            if LOG_PATH.is_file() else {})


def batches(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]


def prompt_generator(idx):
    return torch.Generator(device="cpu").manual_seed(SEED + idx)


# Starting noise for prompt idx, plus its generator with the stream advanced past that
# noise, so the in-loop noise of an ancestral sampler does not repeat the starting noise.
def start_noise(idx, shape):
    generator = prompt_generator(idx)
    return torch.randn((1, *shape), generator=generator), generator


def image_path(config, idx):
    return IMAGES / config / f"{idx:03d}.png"


def save_image(img, path):
    partial = path.with_name(path.stem + ".partial")
    img.convert("RGB").save(partial, format="PNG")
    partial.replace(path)


def free():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def write_log():
    order = [c for c in CONFIG_ORDER if c in log_rows] + [c for c in log_rows if c not in CONFIG_ORDER]
    pd.DataFrame([log_rows[c] for c in order])[LOG_COLUMNS].to_csv(LOG_PATH, index=False)


def render_config(config, meta, render_batch, batch_size=BATCH_SIZE):
    (IMAGES / config).mkdir(parents=True, exist_ok=True)
    todo = [i for i in IDX if not image_path(config, i).exists()]
    print(f"{config}: {len(IDX) - len(todo)} already on disk, {len(todo)} to render")
    if todo:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        started = time.time()
        for part in tqdm(batches(todo, batch_size), desc=config):
            for i, img in zip(part, render_batch(part)):
                save_image(img, image_path(config, i))
        seconds = time.time() - started
        peak = torch.cuda.max_memory_allocated() / 1024 ** 3 if DEVICE.type == "cuda" else float("nan")
        log_rows[config] = {"config": config, **meta,
                            "seconds_total": round(seconds, 1),
                            "sec_per_image": round(seconds / len(todo), 3),
                            "peak_vram_gb": round(peak, 2)}
    elif config not in log_rows:
        log_rows[config] = {"config": config, **meta, "seconds_total": float("nan"),
                            "sec_per_image": float("nan"), "peak_vram_gb": float("nan")}
    log_rows[config]["n_images"] = sum(image_path(config, i).exists() for i in IDX)
    write_log()
    row = log_rows[config]
    print(f"  {row['n_images']} images, {row['sec_per_image']} s/image, peak {row['peak_vram_gb']} GB")

## 4. Text embeddings

SD 1.5 is conditioned through `prompt_embeds`, as in the earlier sweep: CLIP ViT-L/14 runs once in
float32 over all captions and is then freed. Two layers are kept. The last layer is what SD 1.5 was
trained on; clip skip 2 takes the penultimate layer's hidden states and passes them through the
final layer norm. In diffusers' own `clip_skip` argument that is `clip_skip=1`, since its count is
offset by one. The empty-string negative is encoded at the same layer as the prompt.

Captions are encoded one at a time. Batched float32 matmuls differ in the fifth decimal with batch
size, and Euler-ancestral amplifies that into visibly different images, so the guidance notebook's
g = 7.5 would no longer match `sd_tuned`.

In [ ]:
# transformers 4 nests the layer norm under .text_model; transformers 5 does not.
def final_layer_norm(text_encoder):
    return getattr(text_encoder, "text_model", text_encoder).final_layer_norm


@torch.no_grad()
def encode_prompts(texts, tokenizer, text_encoder, layer="last", batch_size=1):
    out = []
    for part in batches(list(texts), batch_size):
        tokens = tokenizer(part, padding="max_length", max_length=CLIP_MAX_TOKENS,
                           truncation=True, return_tensors="pt").to(DEVICE)
        hidden = text_encoder(input_ids=tokens["input_ids"], output_hidden_states=True)
        if layer == "last":
            states = hidden.last_hidden_state
        else:
            states = final_layer_norm(text_encoder)(hidden.hidden_states[-2])
        out.append(states.float().cpu())
    return torch.cat(out)


tokenizer = CLIPTokenizer.from_pretrained(CLIP_SD_MODEL)
text_encoder = CLIPTextModel.from_pretrained(CLIP_SD_MODEL).to(DEVICE, torch.float32).eval()


POS = {layer: encode_prompts(CAPTIONS, tokenizer, text_encoder, layer) for layer in ("last", "penultimate")}
NEG = {layer: encode_prompts([""], tokenizer, text_encoder, layer) for layer in ("last", "penultimate")}

with torch.no_grad():
    check = text_encoder(input_ids=tokenizer(CAPTIONS[:2], padding="max_length", max_length=CLIP_MAX_TOKENS,
                                             truncation=True, return_tensors="pt").input_ids.to(DEVICE),
                         output_hidden_states=True)
    norm_last = final_layer_norm(text_encoder)(check.hidden_states[-1])
    layer_check = float((norm_last - check.last_hidden_state).abs().max())

for layer in POS:
    assert POS[layer].shape == (len(CAPTIONS), CLIP_MAX_TOKENS, 768), POS[layer].shape
    print(f"{layer:<12} prompt {tuple(POS[layer].shape)}  negative {tuple(NEG[layer].shape)}")
print(f"norm(hidden_states[-1]) vs last_hidden_state: max |diff| {layer_check:.1e}")
print(f"last vs penultimate, mean |diff|: {float((POS['last'] - POS['penultimate']).abs().mean()):.3f}")

del text_encoder, tokenizer
free()

## 5. Stable Diffusion 1.5

One pipeline renders all three SD configurations. The baseline is PNDM at 50 steps, SD 1.5's
default; the tuned setting is Euler-ancestral at 30 steps, the best CLIP score in the earlier sweep
(0.3199 against 0.3024). Both use leading timestep spacing and CFG 7.5.

Attention slicing is left off. The earlier notebooks enabled it for the 8 GB laptop GPU, but at
batch 4 PyTorch's built-in attention already peaks under 3 GB, and slicing made each image about
40% slower in a local check. VAE slicing stays on.

In [ ]:
SCHEDULER_CLASSES = {"pndm": PNDMScheduler, "euler_a": EulerAncestralDiscreteScheduler}


def load_sd(repos=SD_REPOS):
    errors = {}
    for repo in repos:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(
                repo, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False)
        except Exception as exc:
            errors[repo] = f"{type(exc).__name__}: {exc}"
            continue
        pipe = pipe.to(DEVICE)
        pipe.set_progress_bar_config(disable=True)
        pipe.vae.enable_slicing()
        return pipe, repo, dict(pipe.scheduler.config)
    raise RuntimeError("no Stable Diffusion repo could be loaded: " + repr(errors))


# Swaps always start from the scheduler config SD 1.5 shipped with, never the live one.
def set_scheduler(pipe, name):
    pipe.scheduler = SCHEDULER_CLASSES[name].from_config(BASE_SCHEDULER_CONFIG,
                                                         timestep_spacing=TIMESTEP_SPACING)


def sd_renderer(pos, neg, steps, guidance, rescale=0.0):
    def render(part):
        noise = [start_noise(i, LATENT_SHAPE) for i in part]
        out = pipe(prompt_embeds=pos[part].to(DEVICE, DTYPE),
                   negative_prompt_embeds=neg.expand(len(part), -1, -1).to(DEVICE, DTYPE),
                   latents=torch.cat([n for n, _ in noise]).to(DEVICE, DTYPE),
                   generator=[g for _, g in noise],
                   num_inference_steps=steps, guidance_scale=guidance, guidance_rescale=rescale,
                   height=SD_SIZE, width=SD_SIZE)
        return out.images
    return render


t0 = time.time()
pipe, SD_REPO, BASE_SCHEDULER_CONFIG = load_sd()
LATENT_SHAPE = (pipe.unet.config.in_channels, SD_SIZE // pipe.vae_scale_factor, SD_SIZE // pipe.vae_scale_factor)
print(f"loaded {SD_REPO} in {time.time() - t0:.1f}s | latents {LATENT_SHAPE} | "
      f"beta_schedule {BASE_SCHEDULER_CONFIG['beta_schedule']}")


for name, cfg in SD_CONFIGS.items():
    set_scheduler(pipe, cfg["sampler"])
    layer = cfg["layer"]
    meta = {"model": SD_REPO,
            "encoder": f"CLIP ViT-L/14 {'last' if layer == 'last' else 'penultimate'} layer",
            "sampler": cfg["sampler"], "steps": cfg["steps"], "guidance": GUIDANCE,
            "resolution": SD_SIZE}
    render_config(name, meta, sd_renderer(POS[layer], NEG[layer], cfg["steps"], GUIDANCE))

del pipe
free()

## 6. LDM-BERT

Latent diffusion conditioned on a BERT-style encoder, at 256 px with its own DDIM scheduler
(eta 0) for 50 steps. Its starting latents come from the same per-prompt generators, and the
pipeline builds the empty-string negative itself.

In [ ]:
ldm = LDMTextToImagePipeline.from_pretrained(LDM_REPO, torch_dtype=DTYPE).to(DEVICE)
ldm.set_progress_bar_config(disable=True)
LDM_SHAPE = (ldm.unet.config.in_channels, LDM_SIZE // 8, LDM_SIZE // 8)
print(f"scheduler {type(ldm.scheduler).__name__} | latents {LDM_SHAPE}")


def ldm_render(part):
    noise = [start_noise(i, LDM_SHAPE) for i in part]
    return ldm(prompt=[CAPTIONS[i] for i in part], height=LDM_SIZE, width=LDM_SIZE,
               num_inference_steps=LDM_STEPS, guidance_scale=GUIDANCE,
               latents=torch.cat([n for n, _ in noise]).to(DEVICE, DTYPE),
               generator=[g for _, g in noise]).images


render_config("ldm_bert", {"model": LDM_REPO, "encoder": "LDM-BERT", "sampler": "ddim",
                           "steps": LDM_STEPS, "guidance": GUIDANCE, "resolution": LDM_SIZE},
              ldm_render)

del ldm
free()

## 7. aMUSEd

A masked-token transformer with the same CLIP ViT-L/14 encoder as SD, unmasking a 256 px image over
12 rounds. It has no starting latent to share. Its pipeline takes a single generator rather than a
list, so it runs one prompt at a time with that prompt's generator; batching would tie an image to
its batch-mates. In a local check that cost about 0.3 s per image over batch 4.

In [ ]:
amused = AmusedPipeline.from_pretrained(AMUSED_REPO, torch_dtype=DTYPE).to(DEVICE)
amused.set_progress_bar_config(disable=True)


def amused_render(part):
    return [amused(prompt=[CAPTIONS[i]], num_inference_steps=AMUSED_STEPS, guidance_scale=GUIDANCE,
                   generator=prompt_generator(i)).images[0]
            for i in part]


render_config("amused", {"model": AMUSED_REPO, "encoder": "CLIP ViT-L/14", "sampler": "masked unmasking",
                         "steps": AMUSED_STEPS, "guidance": GUIDANCE, "resolution": 256},
              amused_render, batch_size=1)

del amused
free()

## 8. Log, preview and timing

The log has one row per configuration. The preview shows the first four prompts through every
configuration, at each model's native resolution scaled to the same tile size.

In [ ]:
def preview_grid(configs, labels, path, idxs):
    fig, axes = plt.subplots(len(idxs), len(configs), figsize=(2.4 * len(configs), 2.5 * len(idxs)),
                             squeeze=False)
    for r, i in enumerate(idxs):
        for c, (config, label) in enumerate(zip(configs, labels)):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if image_path(config, i).exists():
                ax.imshow(Image.open(image_path(config, i)))
            if r == 0:
                ax.set_title(label, fontsize=9)
            if c == 0:
                ax.set_ylabel(textwrap.fill(CAPTIONS[i], 26), rotation=0, ha="right",
                              va="center", fontsize=7, labelpad=8)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


def time_chart(log, path, title):
    fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(log) + 1.4))
    ax.barh(log["config"], log["sec_per_image"], color="#4C72B0")
    ax.invert_yaxis()
    for y, (s, total) in enumerate(zip(log["sec_per_image"], log["seconds_total"])):
        if pd.notna(s):
            ax.text(s, y, f"  {s:.2f} s/img, {total / 60:.1f} min", va="center", fontsize=8)
    ax.set_xlim(0, np.nanmax(log["sec_per_image"].to_numpy(dtype=float)) * 1.45)
    ax.set_xlabel("seconds per image (this session)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


log = pd.read_csv(LOG_PATH)
print(log.to_string(index=False))
print(f"\ntotal rendering time this session: {log['seconds_total'].sum() / 60:.1f} min")

counts = {c: len(list((IMAGES / c).glob("*.png"))) for c in CONFIG_ORDER}
print("images on disk:", counts)

preview_grid(CONFIG_ORDER, CONFIG_ORDER, PLOTS / "m3_render_models_preview.png", IDX[:4])
time_chart(log, PLOTS / "m3_render_models_time.png", f"Rendering cost per configuration ({GPU_NAME})")
print(PLOTS / "m3_render_models_preview.png")
print(PLOTS / "m3_render_models_time.png")

## 9. What was rendered

Five folders of 500 PNGs each under `outputs/images/`: `sd_baseline`, `sd_tuned` and
`sd_tuned_clipskip2` at 512 px, and `ldm_bert` and `amused` at 256 px, one file per prompt named by
its index (`000.png` to `499.png`). `tables/m3_prompts.csv` maps each index to its caption and COCO
image, `tables/m3_generation_log.csv` records the settings and cost of each configuration, and the
preview and timing plots are in `plots/`.

The evaluation notebook computes FID, Inception Score and CLIP score from these files. Attach this
notebook's output to it as a Kaggle input.